# Day 1: Infrastructure Foundation & Iceberg Storage Setup

This notebook simulates the local setup of a mock Iceberg catalog and creates the Bronze raw payload schema with audit metadata (`ingestion_timestamp`, `source_system_id`, `payload_id`, `raw_payload_json`).

In [ ]:
!pip install pyspark pyiceberg duckdb splink -q

In [ ]:
import os
from pyspark.sql import SparkSession

# Set up a local Spark session with Iceberg configurations
spark = SparkSession.builder \
    .appName("Day01_Storage_Setup") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.local.type", "hadoop") \
    .config("spark.sql.catalog.local.warehouse", "/tmp/ehdip_warehouse") \
    .getOrCreate()

print("Spark Session Created with Iceberg Support")

In [ ]:
# Create the Bronze database (namespace)
spark.sql("CREATE NAMESPACE IF NOT EXISTS local.ehdip_bronze")
print("Namespace 'ehdip_bronze' created.")

# Define the Bronze raw payload schema
bronze_ddl = """
CREATE TABLE IF NOT EXISTS local.ehdip_bronze.raw_payloads (
    payload_id STRING,
    source_system_id STRING,
    ingestion_timestamp TIMESTAMP,
    raw_payload_json STRING
)
USING iceberg
PARTITIONED BY (days(ingestion_timestamp))
"""

# Execute DDL
spark.sql(bronze_ddl)
print("Iceberg table 'raw_payloads' created in Bronze layer.")

# Describe table to verify schema
spark.sql("DESCRIBE TABLE local.ehdip_bronze.raw_payloads").show(truncate=False)